# SonicCaption — Environmental Sound Classifier (ESC-50, 10 classes)
**MAI522-5 Speech Processing & Recognition · Member 2 (ML & Recognition)**

This notebook trains the team's own sound-event model:

1. Download ESC-50 (official repo) and keep 10 classes
2. Convert every clip to a 16 kHz mono **log-Mel spectrogram** (fixed parameters saved to `config.json`)
3. Train a compact **4-block CNN** (Conv → BatchNorm → ReLU → MaxPool) with **SpecAugment**-style masking
4. **5-fold cross-validation on the official ESC-50 folds** (test fold *k*, validation fold *k+1*, train on the other 3)
5. Report accuracy, macro precision/recall/F1, per-class F1 and a confusion matrix (out-of-fold, so each clip is tested exactly once)
6. Save `model.pt`, `config.json`, `labels.json`, `metrics.json`, plots and a model card, then upload to the Hugging Face Hub
7. Smoke-test inference with a sliding window, producing timeline sound events

**Colab:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.
**Kaggle:** Settings → Accelerator **GPU T4 x2** (or P100), Internet **On**, add the `HF_TOKEN` secret (Add-ons → Secrets), then **Save Version → Save & Run All**.
The only thing you'll be asked for is your Hugging Face token (step 9). A CPU runtime also works — it's just slower.

## 1. Settings

In [ ]:
import os

# Where checkpoints, metrics and plots are saved (survives Colab disconnects).
# Change this if you want the outputs in a different Drive folder.
DRIVE_OUT_DIR = "/content/drive/MyDrive/SonicCaption_runs"

# The 10 ESC-50 categories we keep (exact names from meta/esc50.csv).
CLASSES = [
    "clapping", "laughing", "coughing", "sneezing", "footsteps",
    "door_wood_knock", "glass_breaking", "siren", "dog", "rain",
]

# Audio / feature parameters — the app MUST use the same values at inference time.
SAMPLE_RATE = 16000
CLIP_SECONDS = 5.0
N_FFT = 400          # 25 ms window
HOP_LENGTH = 160     # 10 ms hop
N_MELS = 64
FMIN, FMAX = 20, 8000

# Training
EPOCHS = int(os.environ.get("SC_EPOCHS", 40))
BATCH_SIZE = 32
LR = 1e-3
WEIGHT_DECAY = 1e-4
SEED = 42
RUN_ALL_FOLDS = True     # False = only the fold-5 run (faster, no cross-validation)

# Hugging Face
PUSH_TO_HUB = os.environ.get("SC_PUSH", "1") == "1"
HF_REPO_NAME = "soniccaption-esc50-cnn"

## 2. Environment, Drive and GPU

In [ ]:
import sys, json, random, time, shutil, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/working")
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_DIR = Path("/content")
elif IN_KAGGLE:
    # Kaggle: everything under /kaggle/working appears in the notebook's Output tab when the run finishes
    WORK_DIR = Path("/kaggle/working")
    DRIVE_OUT_DIR = "/kaggle/working/SonicCaption_runs"
else:
    DRIVE_OUT_DIR = os.environ.get("SC_OUT_DIR", "./runs")
    WORK_DIR = Path(os.environ.get("SC_WORK_DIR", "."))

OUT_DIR = Path(DRIVE_OUT_DIR); OUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR = OUT_DIR / "checkpoints"; CKPT_DIR.mkdir(exist_ok=True)
ART_DIR = OUT_DIR / "hf_model"; ART_DIR.mkdir(exist_ok=True)

import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
import librosa, librosa.display, matplotlib.pyplot as plt

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "no GPU (CPU is fine, just slower)")
print("Outputs ->", OUT_DIR.resolve())

## 3. Download ESC-50 (official repository)

In [ ]:
DATA_ROOT = Path(os.environ.get("SC_DATA_ROOT", WORK_DIR / "ESC-50-master"))
if not (DATA_ROOT / "meta" / "esc50.csv").exists():
    zip_path = WORK_DIR / "esc50.zip"
    print("Downloading ESC-50 (~600 MB)...")
    subprocess.run(["wget", "-q", "-O", str(zip_path),
                    "https://github.com/karolpiczak/ESC-50/archive/master.zip"], check=True)
    subprocess.run(["unzip", "-q", "-o", str(zip_path), "-d", str(WORK_DIR)], check=True)
    zip_path.unlink()
print("ESC-50 at", DATA_ROOT)

## 4. Inspect metadata and keep the 10 classes

In [ ]:
meta = pd.read_csv(DATA_ROOT / "meta" / "esc50.csv")
missing = sorted(set(CLASSES) - set(meta["category"]))
assert not missing, f"Not ESC-50 categories: {missing}. Valid names: {sorted(meta['category'].unique())}"

df = meta[meta["category"].isin(CLASSES)].reset_index(drop=True)
LABEL2ID = {c: i for i, c in enumerate(CLASSES)}
df["y"] = df["category"].map(LABEL2ID)
print(f"{len(df)} clips, {len(CLASSES)} classes")
display(pd.crosstab(df["category"], df["fold"], margins=True)) if (IN_COLAB or IN_KAGGLE) else print(pd.crosstab(df["category"], df["fold"]))

## 5. Log-Mel spectrogram features
Every clip → 16 kHz mono → padded/trimmed to 5 s → 64-band Mel power spectrogram (25 ms Hann window, 10 ms hop) → dB.
Computed once and cached, since augmentation is applied on the spectrogram.

In [ ]:
CLIP_SAMPLES = int(SAMPLE_RATE * CLIP_SECONDS)

def load_audio(path):
    y, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    return y.astype(np.float32)

def fix_length(y, n=CLIP_SAMPLES):
    return y[:n] if len(y) >= n else np.pad(y, (0, n - len(y)))

def log_mel(y):
    S = librosa.feature.melspectrogram(y=y, sr=SAMPLE_RATE, n_fft=N_FFT, hop_length=HOP_LENGTH,
                                       n_mels=N_MELS, fmin=FMIN, fmax=FMAX, power=2.0)
    return librosa.power_to_db(S, ref=1.0, top_db=None).astype(np.float32)  # absolute dB, no per-clip ref

feat_cache = OUT_DIR / f"features_{N_MELS}mel_{SAMPLE_RATE}.npz"
if feat_cache.exists():
    d = np.load(feat_cache); X, y_all, folds = d["X"], d["y"], d["folds"]
    print("Loaded cached features", X.shape)
else:
    t0 = time.time()
    X = np.stack([log_mel(fix_length(load_audio(DATA_ROOT / "audio" / f))) for f in df["filename"]])
    y_all, folds = df["y"].values, df["fold"].values
    np.savez_compressed(feat_cache, X=X, y=y_all, folds=folds)
    print(f"Features {X.shape} in {time.time()-t0:.0f}s")

fig, axes = plt.subplots(2, 5, figsize=(18, 5))
for ax, c in zip(axes.ravel(), CLASSES):
    i = int(np.where(y_all == LABEL2ID[c])[0][0])
    librosa.display.specshow(X[i], sr=SAMPLE_RATE, hop_length=HOP_LENGTH, x_axis="time", y_axis="mel", ax=ax, fmax=FMAX)
    ax.set_title(c); ax.label_outer()
plt.suptitle("Example log-Mel spectrogram per class"); plt.tight_layout()
plt.savefig(OUT_DIR / "example_spectrograms.png", dpi=150); plt.show()

## 6. Dataset with training-only augmentation
- **Random gain**: ±6 dB (a constant offset in the log domain)
- **Time shift**: circular roll of up to ±0.5 s
- **SpecAugment**: 2 frequency masks (≤8 Mel bands) + 2 time masks (≤40 frames)

Normalisation uses the mean/std of the *training* folds only; the values are saved to `config.json`.

In [ ]:
from torch.utils.data import Dataset, DataLoader

class SpecDataset(Dataset):
    def __init__(self, X, y, mean, std, augment=False):
        self.X, self.y, self.mean, self.std, self.augment = X, y, mean, std, augment
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        s = self.X[i].copy()
        if self.augment:
            s = s + np.random.uniform(-6, 6)                                  # gain
            s = np.roll(s, np.random.randint(-50, 51), axis=1)               # time shift (±0.5 s)
            for _ in range(2):                                                # frequency masks
                w = np.random.randint(0, 9); f0 = np.random.randint(0, N_MELS - w + 1)
                s[f0:f0 + w, :] = s.mean()
            for _ in range(2):                                                # time masks
                w = np.random.randint(0, 41); t0 = np.random.randint(0, s.shape[1] - w + 1)
                s[:, t0:t0 + w] = s.mean()
        s = (s - self.mean) / self.std
        return torch.from_numpy(s).unsqueeze(0), int(self.y[i])

## 7. Model — compact 4-block CNN
`[Conv3×3 → BatchNorm → ReLU] ×2 → MaxPool` per block, channels 32 → 64 → 128 → 256, then global average pooling, Dropout(0.3) and a linear classifier.
Global pooling makes the network accept any input length, so the app can run it on 1–5 s windows.

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.MaxPool2d(2))
    def forward(self, x): return self.net(x)

class SonicCNN(nn.Module):
    def __init__(self, n_classes, channels=(32, 64, 128, 256), dropout=0.3):
        super().__init__()
        layers, cin = [], 1
        for c in channels: layers.append(ConvBlock(cin, c)); cin = c
        self.features = nn.Sequential(*layers)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(cin, n_classes)
    def forward(self, x):                     # x: (B, 1, n_mels, frames)
        x = self.features(x)
        x = x.mean(dim=(2, 3))                # global average pooling
        return self.fc(self.dropout(x))

m = SonicCNN(len(CLASSES))
print(f"Parameters: {sum(p.numel() for p in m.parameters()):,}")
print("Output shape:", m(torch.zeros(2, 1, N_MELS, X.shape[2])).shape)

## 8. 5-fold cross-validation on the official folds
For test fold *k*: validation fold = *k+1* (wraps to 1), train on the remaining 3 folds.
The best epoch (by validation accuracy) is kept. Checkpoints are written to Drive every epoch, and finished folds are skipped on re-run, so a Colab disconnect only costs the current fold.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

def run_epoch(model, loader, opt=None):
    train = opt is not None
    model.train(train)
    tot_loss, preds, trues, probs = 0.0, [], [], []
    with torch.set_grad_enabled(train):
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            out = model(xb); loss = F.cross_entropy(out, yb)
            if train: opt.zero_grad(); loss.backward(); opt.step()
            tot_loss += loss.item() * len(yb)
            probs.append(F.softmax(out, 1).detach().cpu().numpy())
            preds += out.argmax(1).tolist(); trues += yb.tolist()
    return tot_loss / len(loader.dataset), accuracy_score(trues, preds), np.array(preds), np.concatenate(probs)

def train_fold(test_fold):
    done_file = CKPT_DIR / f"fold{test_fold}_result.json"
    if done_file.exists():
        print(f"Fold {test_fold}: already done, loading result"); return json.loads(done_file.read_text())
    set_seed(SEED + test_fold)
    val_fold = test_fold % 5 + 1
    tr = ~np.isin(folds, [test_fold, val_fold]); va = folds == val_fold; te = folds == test_fold
    mean, std = float(X[tr].mean()), float(X[tr].std())
    mk = lambda m, aug, sh: DataLoader(SpecDataset(X[m], y_all[m], mean, std, aug), batch_size=BATCH_SIZE, shuffle=sh, num_workers=2)
    dl_tr, dl_va, dl_te = mk(tr, True, True), mk(va, False, False), mk(te, False, False)

    model = SonicCNN(len(CLASSES)).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    hist = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_acc, best_path = -1, CKPT_DIR / f"fold{test_fold}_best.pt"
    for ep in range(1, EPOCHS + 1):
        trl, tra, *_ = run_epoch(model, dl_tr, opt)
        val, vaa, *_ = run_epoch(model, dl_va)
        sched.step()
        for k, v in zip(hist, (trl, val, tra, vaa)): hist[k].append(v)
        if vaa > best_acc:
            best_acc = vaa; torch.save(model.state_dict(), best_path)
        torch.save({"epoch": ep, "model": model.state_dict(), "hist": hist}, CKPT_DIR / f"fold{test_fold}_last.pt")
        if ep == 1 or ep % 5 == 0 or ep == EPOCHS:
            print(f"  fold {test_fold} ep {ep:3d} | train loss {trl:.3f} acc {tra:.3f} | val loss {val:.3f} acc {vaa:.3f}")

    model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    _, te_acc, te_pred, te_prob = run_epoch(model, dl_te)
    res = {"test_fold": test_fold, "val_fold": val_fold, "best_val_acc": best_acc, "test_acc": te_acc,
           "test_macro_f1": f1_score(y_all[te], te_pred, average="macro"),
           "test_idx": np.where(te)[0].tolist(), "test_pred": te_pred.tolist(),
           "norm_mean": mean, "norm_std": std, "history": hist}
    done_file.write_text(json.dumps(res))
    print(f"Fold {test_fold}: test acc {te_acc:.3f} | macro-F1 {res['test_macro_f1']:.3f}")
    return res

results = {}
for k in ([1, 2, 3, 4, 5] if RUN_ALL_FOLDS else [5]):
    t0 = time.time(); results[k] = train_fold(k); print(f"  ({time.time()-t0:.0f}s)")

## 9. Evaluation — metrics, confusion matrix, training curves

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support

idx = np.concatenate([results[k]["test_idx"] for k in results])
pred = np.concatenate([results[k]["test_pred"] for k in results])
true = y_all[idx]

accs = [results[k]["test_acc"] for k in results]; f1s = [results[k]["test_macro_f1"] for k in results]
P, R, F1, _ = precision_recall_fscore_support(true, pred, average="macro", zero_division=0)
per_class = precision_recall_fscore_support(true, pred, labels=range(len(CLASSES)), zero_division=0)

metrics = {
    "protocol": f"ESC-50 official folds; test fold k, val fold k+1, train on remaining 3; {len(results)} fold(s); clip-level (5 s clips)",
    "n_clips_evaluated": int(len(true)),
    "accuracy": float(accuracy_score(true, pred)),
    "macro_precision": float(P), "macro_recall": float(R), "macro_f1": float(F1),
    "fold_accuracy_mean": float(np.mean(accs)), "fold_accuracy_std": float(np.std(accs)),
    "fold_macro_f1_mean": float(np.mean(f1s)), "fold_macro_f1_std": float(np.std(f1s)),
    "per_fold": {int(k): {"accuracy": results[k]["test_acc"], "macro_f1": results[k]["test_macro_f1"]} for k in results},
    "per_class": {c: {"precision": float(per_class[0][i]), "recall": float(per_class[1][i]), "f1": float(per_class[2][i])}
                  for i, c in enumerate(CLASSES)},
}
(ART_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2))
print(classification_report(true, pred, target_names=CLASSES, digits=3, zero_division=0))
print(f"Accuracy {metrics['accuracy']:.3f} | macro-F1 {metrics['macro_f1']:.3f} | "
      f"per-fold acc {metrics['fold_accuracy_mean']:.3f} ± {metrics['fold_accuracy_std']:.3f}")

# Normalised confusion matrix (out-of-fold: every clip tested once)
cm = confusion_matrix(true, pred, labels=range(len(CLASSES)), normalize="true")
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(CLASSES))); ax.set_xticklabels(CLASSES, rotation=45, ha="right")
ax.set_yticks(range(len(CLASSES))); ax.set_yticklabels(CLASSES)
for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        ax.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center", fontsize=8, color="white" if cm[i, j] > 0.5 else "black")
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(f"Normalised confusion matrix ({len(results)}-fold, out-of-fold)")
fig.colorbar(im, fraction=0.046); plt.tight_layout()
plt.savefig(ART_DIR / "confusion_matrix.png", dpi=200); plt.show()

# Training curves (the fold-5 run, which is the published model)
h = results[max(results)]["history"]; ep = range(1, len(h["train_loss"]) + 1)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
a1.plot(ep, h["train_loss"], label="train"); a1.plot(ep, h["val_loss"], label="validation"); a1.set_title("Cross-entropy loss"); a1.set_xlabel("epoch"); a1.legend()
a2.plot(ep, h["train_acc"], label="train"); a2.plot(ep, h["val_acc"], label="validation"); a2.set_title("Accuracy"); a2.set_xlabel("epoch"); a2.legend()
plt.suptitle(f"Training curves — test fold {max(results)}"); plt.tight_layout()
plt.savefig(ART_DIR / "training_curves.png", dpi=200); plt.show()

## 10. Save the model package
The published model is the **test-fold-5 run** (best validation epoch). It was never trained on fold 5, so its fold-5 score is an honest held-out result.
`config.json` holds everything the app needs to reproduce the exact preprocessing.

In [ ]:
final = results[max(results)]
model = SonicCNN(len(CLASSES))
model.load_state_dict(torch.load(CKPT_DIR / f"fold{final['test_fold']}_best.pt", map_location="cpu"))
torch.save(model.state_dict(), ART_DIR / "model.pt")

config = {
    "model_name": "soniccaption-esc50-cnn",
    "architecture": {"type": "SonicCNN", "channels": [32, 64, 128, 256], "dropout": 0.3, "n_classes": len(CLASSES)},
    "sample_rate": SAMPLE_RATE, "clip_seconds": CLIP_SECONDS,
    "n_fft": N_FFT, "hop_length": HOP_LENGTH, "n_mels": N_MELS, "fmin": FMIN, "fmax": FMAX,
    "window": "hann", "power": 2.0, "db_ref": 1.0,
    "normalization": {"mean": final["norm_mean"], "std": final["norm_std"]},
    "published_run": {"test_fold": final["test_fold"], "val_fold": final["val_fold"], "test_accuracy": final["test_acc"], "test_macro_f1": final["test_macro_f1"]},
    "training": {"epochs": EPOCHS, "batch_size": BATCH_SIZE, "optimizer": "Adam", "lr": LR, "weight_decay": WEIGHT_DECAY, "scheduler": "cosine", "seed": SEED,
                 "augmentation": ["gain ±6 dB", "time roll ±0.5 s", "2 freq masks ≤8 bins", "2 time masks ≤40 frames"]},
}
(ART_DIR / "config.json").write_text(json.dumps(config, indent=2))
(ART_DIR / "labels.json").write_text(json.dumps({"id2label": {i: c for i, c in enumerate(CLASSES)}, "label2id": LABEL2ID}, indent=2))

# Standalone model definition so the backend can import it
(ART_DIR / "model_def.py").write_text("""import torch.nn as nn

class ConvBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.MaxPool2d(2))
    def forward(self, x): return self.net(x)

class SonicCNN(nn.Module):
    def __init__(self, n_classes, channels=(32, 64, 128, 256), dropout=0.3):
        super().__init__()
        layers, cin = [], 1
        for c in channels:
            layers.append(ConvBlock(cin, c)); cin = c
        self.features = nn.Sequential(*layers)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(cin, n_classes)
    def forward(self, x):
        x = self.features(x).mean(dim=(2, 3))
        return self.fc(self.dropout(x))
""")

pc = "\n".join(f"| {c} | {v['precision']:.3f} | {v['recall']:.3f} | {v['f1']:.3f} |" for c, v in metrics["per_class"].items())
(ART_DIR / "README.md").write_text(f"""---
license: cc-by-nc-3.0
tags: [audio-classification, environmental-sound, esc-50, pytorch]
---
# SonicCaption ESC-50 CNN (10 classes)

Compact CNN that classifies short audio windows into 10 environmental sound classes. Trained by the SonicCaption team
(MAI522-5 Speech Processing & Recognition) to add non-speech events such as [Applause] or [Door knock] to captions.

## Classes
{", ".join(CLASSES)}

## Data
[ESC-50](https://github.com/karolpiczak/ESC-50) (K. J. Piczak, *ESC: Dataset for Environmental Sound Classification*, ACM MM 2015),
10-class subset, 40 clips per class, 5 s each. ESC-50 is licensed CC BY-NC 3.0, so this model is for non-commercial use.

## Input
16 kHz mono audio → log-Mel spectrogram ({N_MELS} Mel bands, {N_FFT}-sample Hann window, {HOP_LENGTH}-sample hop, power→dB with ref 1.0)
→ normalised with the mean/std in `config.json`.

## Architecture
4 conv blocks ([Conv3×3-BN-ReLU]×2 + MaxPool2), channels 32/64/128/256, global average pooling, Dropout 0.3, linear head.

## Training
Adam (lr {LR}, weight decay {WEIGHT_DECAY}), cosine schedule, {EPOCHS} epochs, batch {BATCH_SIZE}, best epoch by validation accuracy.
Augmentation (training only): random gain, time shift, SpecAugment-style frequency and time masks.

## Evaluation
{metrics["protocol"]}.

| Metric | Value |
|---|---|
| Accuracy (out-of-fold) | {metrics["accuracy"]:.3f} |
| Macro precision | {metrics["macro_precision"]:.3f} |
| Macro recall | {metrics["macro_recall"]:.3f} |
| Macro F1 | {metrics["macro_f1"]:.3f} |
| Per-fold accuracy | {metrics["fold_accuracy_mean"]:.3f} ± {metrics["fold_accuracy_std"]:.3f} |

| Class | Precision | Recall | F1 |
|---|---|---|---|
{pc}

The published weights are from the run with test fold {final["test_fold"]} (test accuracy {final["test_acc"]:.3f}).

## Limitations
- Only 400 training clips; real-world recordings differ from ESC-50 (domain shift).
- Closed set of 10 classes: any other sound is forced into one of them, so the app applies a confidence threshold.
- Trained on 5 s clips with one sound each; overlapping sounds and speech-plus-event mixtures are not modelled.
- Sliding-window timestamps are approximate, not exact event boundaries.
""")
print("Saved:", sorted(p.name for p in ART_DIR.iterdir()))

## 11. Upload to the Hugging Face Hub
You'll be asked for a **write** token (huggingface.co → Settings → Access Tokens).
If you've saved it as a Colab secret called `HF_TOKEN` (🔑 icon in the left sidebar), it's used automatically.

In [ ]:
if PUSH_TO_HUB:
    from huggingface_hub import HfApi, login
    token = os.environ.get("HF_TOKEN")
    if not token and IN_COLAB:
        try:
            from google.colab import userdata; token = userdata.get("HF_TOKEN")
        except Exception:
            pass
    if not token and IN_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient; token = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            pass
    if not token and IN_COLAB:
        from getpass import getpass; token = getpass("Hugging Face write token: ")
    if not token:
        raise RuntimeError("No Hugging Face token found. Add a secret named HF_TOKEN "
                           "(Kaggle: Add-ons -> Secrets; Colab: the key icon) and re-run this cell. "
                           "Your trained model is already saved in " + str(ART_DIR))
    login(token=token)
    api = HfApi()
    repo_id = f"{api.whoami()['name']}/{HF_REPO_NAME}"
    api.create_repo(repo_id, exist_ok=True, repo_type="model")
    commit = api.upload_folder(folder_path=str(ART_DIR), repo_id=repo_id, commit_message="Upload SonicCaption ESC-50 CNN")
    revision = commit.oid
    print("Model:", f"https://huggingface.co/{repo_id}")
    print("Pin this revision in the app config:", revision)
    (OUT_DIR / "hf_info.json").write_text(json.dumps({"repo_id": repo_id, "revision": revision}, indent=2))
else:
    print("PUSH_TO_HUB is False — skipped.")

## 12. Inference smoke test — sliding window → timeline sound events
This is the same logic the backend's `sound_classifier.py` will use. It builds a 20 s test track from **fold-5 clips the published model never trained on**,
slides a 1 s window with 0.5 s hop, applies a confidence threshold and merges neighbouring windows with the same label.

In [ ]:
def load_package(path):
    cfg = json.loads((path / "config.json").read_text())
    labels = json.loads((path / "labels.json").read_text())["id2label"]
    net = SonicCNN(cfg["architecture"]["n_classes"]); net.load_state_dict(torch.load(path / "model.pt", map_location="cpu")); net.eval()
    return net, cfg, [labels[str(i)] if str(i) in labels else labels[i] for i in range(len(labels))]

def classify_audio_window(net, cfg, labels, audio):
    S = librosa.power_to_db(librosa.feature.melspectrogram(y=audio, sr=cfg["sample_rate"], n_fft=cfg["n_fft"], hop_length=cfg["hop_length"],
                            n_mels=cfg["n_mels"], fmin=cfg["fmin"], fmax=cfg["fmax"], power=2.0), ref=1.0, top_db=None)
    S = (S - cfg["normalization"]["mean"]) / cfg["normalization"]["std"]
    with torch.no_grad():
        p = F.softmax(net(torch.tensor(S, dtype=torch.float32)[None, None]), 1)[0].numpy()
    order = np.argsort(-p)
    return [{"label": labels[i], "score": float(p[i])} for i in order]

def detect_events(net, cfg, labels, audio, win=1.0, hop=0.5, threshold=0.5):
    sr, W, H = cfg["sample_rate"], int(win * cfg["sample_rate"]), int(hop * cfg["sample_rate"])
    events = []
    for s in range(0, max(len(audio) - W, 0) + 1, H):
        seg = audio[s:s + W]
        if np.sqrt(np.mean(seg ** 2)) < 1e-3:            # skip near-silent windows
            continue
        top = classify_audio_window(net, cfg, labels, seg)[0]
        if top["score"] < threshold:
            continue
        st, en = s / sr, (s + W) / sr
        if events and events[-1]["label"] == top["label"] and st <= events[-1]["end"]:
            events[-1]["end"] = en; events[-1]["confidence"] = max(events[-1]["confidence"], round(top["score"], 3))
        else:
            events.append({"start": round(st, 2), "end": round(en, 2), "label": top["label"],
                           "confidence": round(top["score"], 3), "model": "soniccaption-esc50-cnn"})
    return events

net, cfg, labels = load_package(ART_DIR)
test_rows = df[df["fold"] == final["test_fold"]].groupby("category").head(1).head(4)
track = np.concatenate([fix_length(load_audio(DATA_ROOT / "audio" / f)) for f in test_rows["filename"]])
print("Ground truth:", [f"{i*5}-{i*5+5}s: {c}" for i, c in enumerate(test_rows["category"])])
for e in detect_events(net, cfg, labels, track):
    print(e)